In [1]:
import numpy as np

from sklearn.datasets import make_regression, make_friedman1, make_classification
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn import metrics

from selector import OrthogonalSelector, distance_correlation

# Regression

## Simple linear dataset

In [2]:
x, y, coeff = make_regression(n_samples=1500, n_features=20, n_informative=10, noise=1, random_state=42, coef=True, shuffle=True)

# Rust enforces float32 due to performance reasons
x = x.astype(np.float32)
y = y.astype(np.float32)

x_tr, x_te, y_tr, y_te = train_test_split(x, y, test_size=500, random_state=42)

selector = OrthogonalSelector(
    fixed_feature_indices=[],
    score_type="squared_partial_correlation",
    min_score=1e-2,
    center_features=True,
    n_jobs=10
)

selected_linear_features, scores = selector.fit(
    x=x_tr,
    y=y_tr.reshape((-1,1))
)

Round 0; features to explore: 20; best feature idx/score: 17/0.23677
Round 1; features to explore: 19; best feature idx/score: 3/0.26923
Round 2; features to explore: 18; best feature idx/score: 2/0.30145
Round 3; features to explore: 17; best feature idx/score: 0/0.32073
Round 4; features to explore: 16; best feature idx/score: 9/0.38363
Round 5; features to explore: 15; best feature idx/score: 18/0.38839
Round 6; features to explore: 14; best feature idx/score: 7/0.49161
Round 7; features to explore: 13; best feature idx/score: 1/0.80887
Round 8; features to explore: 12; best feature idx/score: 14/0.84943
Round 9; features to explore: 11; best feature idx/score: 12/0.65215
Round 10; features to explore: 10; best feature idx/score: 16/0.00028


In [3]:
pipe = make_pipeline(
    ColumnTransformer([("scaler", StandardScaler(), selected_linear_features)], remainder="drop"),
    LinearRegression()
)

pipe.fit(x_tr, y_tr)

print(f"R^2: {metrics.r2_score(y_te, pipe.predict(x_te)):.2f}")

R^2: 1.00


## Non-linear dataset

In [4]:
x, y = make_friedman1(n_samples=1500, n_features=20, noise=1.0, random_state=42)
# Rust enforces float32 due to performance reasons
x = x.astype(np.float32)
y = y.astype(np.float32)

x_tr, x_te, y_tr, y_te = train_test_split(x, y, test_size=500, random_state=42)

selector = OrthogonalSelector(
    fixed_feature_indices=[],
    score_type="squared_partial_correlation",
    min_score=1e-2,
    center_features=True,
    n_jobs=10
)

selected_linear_features, scores = selector.fit(
    x=x_tr,
    y=y_tr.reshape((-1,1))
)

Round 0; features to explore: 20; best feature idx/score: 3/0.03956
Round 1; features to explore: 19; best feature idx/score: 1/0.01758
Round 2; features to explore: 18; best feature idx/score: 0/0.01569
Round 3; features to explore: 17; best feature idx/score: 4/0.00988


In [5]:
pipe = make_pipeline(
    ColumnTransformer([("scaler", StandardScaler(), selected_linear_features)], remainder="drop"),
    LinearRegression()
)

pipe.fit(x_tr, y_tr)
pipe_metric = metrics.r2_score(y_te, pipe.predict(x_te))

base_est = make_pipeline(StandardScaler(), HistGradientBoostingRegressor())
base_est.fit(x_tr, y_tr)
base_metric = metrics.r2_score(y_te, base_est.predict(x_te))

print(f"""
Base estimator ({base_est.steps[-1][1].__class__}): {base_metric:.4f};
Only linear features: {pipe_metric:.4f}
""")

y_resid = y_tr - pipe.predict(x_tr)


Base estimator (<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingRegressor'>): 0.9058;
Only linear features: 0.6274



### Adding nonlinearities

In [6]:
dcor = distance_correlation(
    x=x_tr.astype(np.float32),
    y=y_resid.reshape((-1,1)).astype(np.float32),
    n_jobs=10
)

nonlinear_features = np.where(dcor > 1e-1)[0].tolist()
all_features = list(set(selected_linear_features + nonlinear_features))

Progress: 1/20
Progress: 2/20
Progress: 3/20
Progress: 4/20
Progress: 5/20
Progress: 6/20
Progress: 7/20
Progress: 8/20
Progress: 9/20
Progress: 10/20
Progress: 11/20
Progress: 12/20
Progress: 13/20
Progress: 14/20
Progress: 15/20
Progress: 16/20
Progress: 17/20
Progress: 18/20
Progress: 19/20
Progress: 20/20


In [7]:
pipe = make_pipeline(
    ColumnTransformer([
            ("scaler", StandardScaler(), all_features),
            ("kernel", RBFSampler(n_components=50, gamma=.1), nonlinear_features)
        ],
        remainder="drop"
    ),
    LinearRegression()
)

pipe.fit(x_tr, y_tr)
pipe_metric = metrics.r2_score(y_te, pipe.predict(x_te))

print(f"""
Base estimator ({base_est.steps[-1][1].__class__}): {base_metric:.4f};
Only linear features: {pipe_metric:.4f}
""")


Base estimator (<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingRegressor'>): 0.9058;
Only linear features: 0.9564



# Classification

## Binary

In [8]:
METRIC = lambda y_true, y_pred: metrics.f1_score(y_true, y_pred)

x, y = make_classification(
    n_samples=1500,
    n_features=20,
    n_informative=10,
    n_repeated=3,
    n_redundant=4,
    flip_y=0.07,
    random_state=42,
    shuffle=True
)
# Rust enforces float32 due to performance reasons
x = x.astype(np.float32)
y = y.astype(np.float32)

x_tr, x_te, y_tr, y_te = train_test_split(x, y, test_size=500, random_state=42)

selector = OrthogonalSelector(
    fixed_feature_indices=[],
    score_type="logit_gradient",
    min_score=1e-3,
    center_features=True,
    n_jobs=10
)

selected_linear_features, scores = selector.fit(
    x=x_tr,
    y=y_tr.reshape((-1,1))
)

pipe = make_pipeline(
    ColumnTransformer([("scaler", StandardScaler(), selected_linear_features)], remainder="drop"),
    LogisticRegression(C=10.0, penalty="l2", max_iter=1_000, n_jobs=10)
)

base_est = make_pipeline(StandardScaler(), HistGradientBoostingClassifier())
base_est.fit(x_tr, y_tr)
base_pred = base_est.predict(x_te)

base_metric = METRIC(y_te, base_pred)

pipe.fit(x_tr, y_tr)
pipe_pred = pipe.predict(x_te)
pipe_metric = METRIC(y_te, pipe_pred)

print(f"""
Base estimator ({base_est.steps[-1][1].__class__}): {base_metric:.4f};
Only linear features: {pipe_metric:.4f}
""")

Round 0; features to explore: 20; best feature idx/score: 14/0.69314
Round 1; features to explore: 19; best feature idx/score: 5/0.07494
Round 2; features to explore: 18; best feature idx/score: 19/0.05092
Round 3; features to explore: 17; best feature idx/score: 1/0.03109
Round 4; features to explore: 16; best feature idx/score: 11/0.06258
Round 5; features to explore: 15; best feature idx/score: 3/0.01808
Round 6; features to explore: 14; best feature idx/score: 17/0.01113
Round 7; features to explore: 13; best feature idx/score: 13/0.00558
Round 8; features to explore: 12; best feature idx/score: 18/0.00017

Base estimator (<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingClassifier'>): 0.8806;
Only linear features: 0.8180



In [9]:
dcor = distance_correlation(
    x=x_tr,
    y=y_tr.reshape((-1,1)),
    n_jobs=10
)

nonlinear_features = np.where(dcor > 1e-1)[0].tolist()

all_features = list(set(selected_linear_features + nonlinear_features))

Progress: 1/20
Progress: 2/20
Progress: 3/20
Progress: 4/20
Progress: 5/20
Progress: 6/20
Progress: 7/20
Progress: 8/20
Progress: 11/20
Progress: 9/20
Progress: 10/20
Progress: 13/20
Progress: 12/20
Progress: 14/20
Progress: 15/20
Progress: 16/20
Progress: 17/20
Progress: 18/20
Progress: 19/20
Progress: 20/20


In [10]:
pipe = make_pipeline(
    ColumnTransformer([
            ("scaler", StandardScaler(), all_features),
            ("kernel", RBFSampler(n_components=100, gamma=.1, random_state=42), nonlinear_features)
        ],
        remainder="drop"
    ),
    LogisticRegression(C=1.0, penalty="l2", max_iter=1_000, n_jobs=10)
)

pipe.fit(x_tr, y_tr)
pipe_pred = pipe.predict(x_te)
pipe_metric = METRIC(y_te, pipe_pred)

print(f"""
Base estimator ({base_est.steps[-1][1].__class__}): {base_metric:.4f};
Full model: {pipe_metric:.4f}
""")


Base estimator (<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingClassifier'>): 0.8806;
Full model: 0.8462



## Multi-class

In [11]:
METRIC = lambda y_true, y_pred: metrics.f1_score(y_true, y_pred, average="weighted")

x, y = make_classification(
    n_classes=4,
    n_samples=1500,
    n_features=20,
    n_informative=10,
    n_repeated=3,
    n_redundant=4,
    flip_y=0.07,
    random_state=42,
    shuffle=True
)
# Rust enforces float32 due to performance reasons
x = x.astype(np.float32)
y = y.astype(np.float32)

x_tr, x_te, y_tr, y_te = train_test_split(x, y, test_size=500, random_state=42)

selector = OrthogonalSelector(
    fixed_feature_indices=[],
    score_type="logit_gradient",
    min_score=1e-2,
    center_features=True,
    n_jobs=10
)

selected_linear_features, scores = selector.fit(
    x=x_tr,
    y=y_tr.reshape((-1,1))
)

pipe = make_pipeline(
    ColumnTransformer([("scaler", StandardScaler(), selected_linear_features)], remainder="drop"),
    LogisticRegression(C=10.0, penalty="l2", max_iter=1_000, n_jobs=10)
)

base_est = make_pipeline(StandardScaler(), HistGradientBoostingClassifier())
base_est.fit(x_tr, y_tr)
base_pred = base_est.predict(x_te)

base_metric = METRIC(y_te, base_pred)

pipe.fit(x_tr, y_tr)
pipe_pred = pipe.predict(x_te)
pipe_metric = METRIC(y_te, pipe_pred)

print(f"""
Base estimator ({base_est.steps[-1][1].__class__}): {base_metric:.4f};
Only linear features: {pipe_metric:.4f}
""")

Round 0; features to explore: 20; best feature idx/score: 2/2.26803
Round 1; features to explore: 19; best feature idx/score: 13/0.12725
Round 2; features to explore: 18; best feature idx/score: 12/0.09826
Round 3; features to explore: 17; best feature idx/score: 1/0.07907
Round 4; features to explore: 16; best feature idx/score: 8/0.07172
Round 5; features to explore: 15; best feature idx/score: 18/0.07863
Round 6; features to explore: 14; best feature idx/score: 11/0.04761
Round 7; features to explore: 13; best feature idx/score: 14/0.03056
Round 8; features to explore: 12; best feature idx/score: 6/0.02674
Round 9; features to explore: 11; best feature idx/score: 16/0.01362
Round 10; features to explore: 10; best feature idx/score: 7/0.00644

Base estimator (<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingClassifier'>): 0.7308;
Only linear features: 0.6079



In [12]:
dcor = distance_correlation(
    x=x_tr,
    y=y_tr.reshape((-1,1)),
    n_jobs=10
)

nonlinear_features = np.where(dcor > 1e-1)[0].tolist()

all_features = list(set(selected_linear_features + nonlinear_features))

pipe = make_pipeline(
    ColumnTransformer([
            ("scaler", StandardScaler(), all_features),
            ("kernel", RBFSampler(n_components=100, gamma=0.01, random_state=42), nonlinear_features)
        ],
        remainder="drop"
    ),
    LogisticRegression(C=1.0, penalty="l2", max_iter=1_000, n_jobs=10)
)

pipe.fit(x_tr, y_tr)
pipe_pred = pipe.predict(x_te)
pipe_metric = METRIC(y_te, pipe_pred)

print(f"""
Base estimator ({base_est.steps[-1][1].__class__}): {base_metric:.4f};
Full model: {pipe_metric:.4f}
""")

Progress: 1/20
Progress: 2/20
Progress: 3/20
Progress: 4/20
Progress: 5/20
Progress: 6/20
Progress: 7/20
Progress: 8/20
Progress: 9/20
Progress: 10/20
Progress: 11/20
Progress: 12/20
Progress: 13/20
Progress: 14/20
Progress: 15/20
Progress: 16/20
Progress: 17/20
Progress: 18/20
Progress: 19/20
Progress: 20/20

Base estimator (<class 'sklearn.ensemble._hist_gradient_boosting.gradient_boosting.HistGradientBoostingClassifier'>): 0.7308;
Full model: 0.6904

